In [ ]:
import torch
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])


train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

In [2]:
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=False
)

In [5]:
import torch.nn as nn
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.hidden_layer = nn.Linear(in_features=784, out_features=128)
    self.relu = nn.ReLU()
    self.output_layer = nn.Linear(in_features=128, out_features=10)
  def forward(self, x):
    x = x.view(-1, 784)
    x = self.hidden_layer(x)
    x = self.relu(x)
    x = self.output_layer(x)
    return x

In [6]:
model = NeuralNetwork()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [8]:
epochs = 5

for epoch in range(epochs):
    running_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()
        predictions = model(images)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch + 1}/{epochs} - Loss: {running_loss / len(train_loader):.4f}")

Epoch 1/5 - Loss: 0.0604
Epoch 2/5 - Loss: 0.0470
Epoch 3/5 - Loss: 0.0412
Epoch 4/5 - Loss: 0.0302
Epoch 5/5 - Loss: 0.0283


In [11]:
def predict_digit(model, image_tensor):
  model.eval()
  with torch.no_grad():
    raw_output = model(image_tensor)
    probabilities = torch.softmax(raw_output, dim=1)
    predicted_class = torch.argmax(probabilities, dim=1)
    return predicted_class.item()

In [12]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        predicted = torch.argmax(outputs, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on test set: {accuracy:.2f}%")

Accuracy on test set: 97.85%
